# Pod H XGBoost Pipeline Summary

This notebook organizes the XGBoost work in a step-by-step format so the model flow, validation results, and forecast output can be reviewed cell by cell.

## Current Results

- Target: Revenue
- Backtest setup: 30-day horizon, 4 folds, recursive rollout
- Avg MAPE: 6.50%
- Avg within 10%: 83.33%
- 30-day forecast: generated with 85% CI

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'projects').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root with a projects/ folder.')

root = find_repo_root()
metrics_path = root / 'projects' / 'synthetic_ecommerce_data' / 'outputs' / 'backtests' / 'revenue' / 'metrics.json'
forecast_path = root / 'projects' / 'synthetic_ecommerce_data' / 'outputs' / 'forecasts' / 'revenue' / 'forecast_30d.csv'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
forecast = pd.read_csv(forecast_path)

summary_df = pd.DataFrame([metrics['summary']])
print('=== SUMMARY ===')
display(summary_df)

print('=== FULL 30-DAY FORECAST (ALL ROWS) ===')
display(forecast)

print('=== RAW SUMMARY JSON ===')
print(json.dumps(metrics['summary'], indent=2, ensure_ascii=False))

## 2) Backtest breakdown

The rolling backtest uses 4 folds with a 30-day test horizon each.
This table shows the fold-level behavior that leads to the final average score.

In [ ]:
fold_rows = []
for fold in metrics['folds']:
    fold_rows.append(
        {
            'fold': fold['fold'],
            'mape': fold['mape'],
            'within_10': fold['within_10'],
            'n_test': fold['n_test'],
            'test_start': fold['test_start'],
            'test_end': fold['test_end'],
            'max_abs_error': max(fold['abs_errors']),
            'mean_abs_error': sum(fold['abs_errors']) / len(fold['abs_errors']),
        }
    )

folds_df = pd.DataFrame(fold_rows)
print('=== FOLD-LEVEL SUMMARY ===')
display(folds_df)

print('=== ALL ABSOLUTE ERRORS BY FOLD (FULL LIST) ===')
for fold in metrics['folds']:
    print(f"Fold {fold['fold']} | n={len(fold['abs_errors'])}")
    print(fold['abs_errors'])
    print('-' * 80)

## 3) Files used by the pipeline

These are the core scripts behind the XGBoost workflow:

- `rebuild_xgboost_pipeline/utils.py`
- `rebuild_xgboost_pipeline/build_features.py`
- `rebuild_xgboost_pipeline/backtest.py`
- `rebuild_xgboost_pipeline/forecast_30d.py`

Reference docs in the same bundle:

- `docs/XGBOOST_PIPELINE_SUMMARY.html`
- `docs/XGBOOST_VALIDATION_REPORT.md`
- `docs/REVENUE_FOCUS_CHECKLIST.md`

## 4) How to reproduce

Run the pipeline in this order:

```bash
/opt/homebrew/bin/python3.13 rebuild_xgboost_pipeline/build_features.py
/opt/homebrew/bin/python3.13 rebuild_xgboost_pipeline/backtest.py --target Revenue
/opt/homebrew/bin/python3.13 rebuild_xgboost_pipeline/forecast_30d.py --target Revenue
```